# 🧠 Fine-Tuning SciAssistant (SLM) using Unsloth & QLoRA

This document outlines the complete pipeline for fine-tuning our Small Language Model (SciAssistant). It covers data preparation, QLoRA training, GGUF quantization, and pushing the final model to the Hugging Face Hub.

## Step 1: Data Acquisition, Preprocessing & Structuring
*In this step, we load the `SciInstruct` dataset, dynamically map the instruction and response columns, sample 35,000 high-quality records, and format them into a strict Alpaca prompt template suitable for instruction-tuning.*

In [ ]:
# تثبيت المكتبات المطلوبة
!pip install datasets pandas -q

import pandas as pd
from datasets import load_dataset

print("⏳ جاري تحميل قاعدة بيانات SciInstruct...")

# 1. تحميل البيانات
dataset = load_dataset("zd21/SciInstruct", split="train")
df = dataset.to_pandas()

# 2. اكتشاف أسماء الأعمدة تلقائياً (الحل الجذري لمشكلة الملف الفاضي)
columns = df.columns.tolist()
print(f"🔍 الأعمدة الموجودة في قاعدة البيانات هي: {columns}")

# نبحث عن العمود الذي يحتوي على السؤال، ثم العمود الذي يحتوي على الإجابة
input_col = next((col for col in ['problem', 'question', 'instruction', 'query', 'input'] if col in columns), columns[0])
output_col = next((col for col in ['solution', 'answer', 'output', 'response'] if col in columns), columns[1])

print(f"✅ تم تحديد الأعمدة: السؤال من ({input_col}) والإجابة من ({output_col})")

# 3. زيادة حجم العينة لنتائج احترافية ("شغل زين")
# نسحب 35,000 صف كعينة تدريبية قوية جداً
SAMPLE_SIZE = 35000
df_sampled = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print("⚙️ جاري إعادة هيكلة البيانات لصيغة Unsloth...")

# 4. بناء الهيكل الصارم (Alpaca Format)
formatted_data = []
SYSTEM_PROMPT = "You are an expert AI Research Assistant. Answer the scientific question accurately and thoroughly based on academic reasoning."

for index, row in df_sampled.iterrows():
    question = str(row[input_col]).strip()
    answer = str(row[output_col]).strip()
    
    # فلترة سريعة: نتخطى أي صف فارغ أو يحتوي على سؤال قصير جداً
    if not question or not answer or len(question) < 10:
        continue
        
    formatted_data.append({
        "instruction": SYSTEM_PROMPT,
        "input": question,
        "output": answer
    })

# 5. حفظ البيانات
formatted_df = pd.DataFrame(formatted_data)
output_filename = "scientific_assistant_dataset_35k.jsonl"
formatted_df.to_json(output_filename, orient="records", lines=True)

print(f"🎉 اكتمل العمل بنجاح! تم حفظ {len(formatted_df)} مثال أكاديمي في ملف: {output_filename}")
print("هذا الحجم ممتاز جداً وسيعطيك أداءً يقارب الموديلات المغلقة في فهم الأبحاث.")

## Step 2: Environment Setup & Kernel Restart
*We install the `unsloth` library and its dependencies to enable fast, 4-bit quantized training. A kernel restart is required afterward to ensure no memory is leaked before training begins.*

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl>=0.18.2,<=0.24.0" peft accelerate bitsandbytes

In [ ]:
import os
os.kill(os.getpid(), 9)

## Step 3: Pre-Training System Sanity Checks
*Before initializing the model, we verify that our processed dataset survived the kernel restart and monitor the available GPU memory to prevent Out-Of-Memory (OOM) failures.*

In [ ]:
import os

file_name = "scientific_assistant_dataset_35k.jsonl"

# فحص هل الملف موجود في مسار العمل الحالي أو لا
if os.path.exists(file_name):
    print(f"✅ ممتاز! الملف '{file_name}' موجود وجاهز.")
    print("تقدر تتوكل على الله وتشغل كود التدريب (الخلية الطويلة) الآن.")
else:
    print(f"❌ انتبه! الملف '{file_name}' غير موجود. يبدو أن السيرفر حذفه لما سوى ريستارت.")
    print("الحل: ارجع شغل الخلية القديمة (حقت تجهيز البيانات من HuggingFace) عشان يصنع الملف من جديد، وبعدين شغل كود التدريب.")

In [1]:
import torch
print(f"الذاكرة المتاحة: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

الذاكرة المتاحة: 14.5 GB


## Step 4: Model Initialization, LoRA Configuration & Fine-Tuning
*We load the LLaMA-3.2-3B model in 4-bit precision, apply LoRA adapters to the target attention and MLP modules, and execute the instruction-tuning process using the `SFTTrainer`.*

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

max_seq_length = 2048
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",  # 3B بدل 8B
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
    device_map = {"": 0},
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# باقي الكود نفسه بدون تغيير

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
### Instruction:
{}
### Input:
{}
### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token 

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

print("⚙️ جاري تحميل بيانات التدريب...")
dataset = load_dataset("json", data_files="scientific_assistant_dataset_35k.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

print("🚀 بدء عملية الضبط الدقيق...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
    args = TrainingArguments(
        per_device_train_batch_size = 1,  # كان 2
        gradient_accumulation_steps = 8,  # كان 4 - نعوّض batch size
        warmup_steps = 5,
        max_steps = 500, 
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", 
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 2,
        dataloader_pin_memory = False,  # يوفر ذاكرة إضافية
    ),
)

trainer_stats = trainer.train()
print("🎉 انتهى التدريب بنجاح!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/34997 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 34,997 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,0.966652
2,1.292116
3,2.255617
4,1.093231
5,1.386919
6,1.663637
7,2.383426
8,1.531950
9,1.284921
10,1.077714


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.


🎉 انتهى التدريب بنجاح!


## Step 5: Local Model Saving & GGUF Quantization
*After successful training, we save the LoRA adapters locally. Then, we compress and quantize the model into the GGUF format (`q4_k_m`) to allow highly efficient local execution on standard consumer hardware.*

In [3]:
# حفظ نسخة LoRA الكاملة (الأوزان المدربة فقط)
model.save_pretrained("scientific_assistant_lora")
tokenizer.save_pretrained("scientific_assistant_lora")

print("✅ تم حفظ النسخة الكاملة (LoRA) بنجاح في مجلد: scientific_assistant_lora")

Unsloth: Restored added_tokens_decoder metadata in scientific_assistant_lora/tokenizer_config.json.


✅ تم حفظ النسخة الكاملة (LoRA) بنجاح في مجلد: scientific_assistant_lora


In [4]:
# ضغط وتحويل الموديل لصيغة GGUF للتشغيل على المعالج والرامات محلياً
model.save_pretrained_gguf(
    "scientific_assistant_gguf", 
    tokenizer, 
    quantization_method = "q4_k_m"
)

print("✅ تم تحويل الموديل للنسخة المختصرة بنجاح في مجلد: scientific_assistant_gguf")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in scientific_assistant_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:46<00:00, 23.01s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/scientific_assistant_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['scientific_assistant_gguf_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['scientific_assistant_gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model scientific_assistant_gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to scientific_assistant_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f scientific_assistant_gguf_gguf/Modelfile
✅ تم تحويل الموديل للنسخة المختصرة بنجاح في مجلد: scientific_assistant_gguf


## Step 6: Hugging Face Authentication & Cloud Deployment
*Finally, we authenticate with Hugging Face and push both the full LoRA weights and the optimized GGUF model directly to the cloud repository (Marwan915) for seamless integration into our multi-agent workflow.*

In [5]:
from huggingface_hub import login

# سيطلب منك الـ Token الآن
login()

In [6]:
!git config --global credential.helper store

In [1]:
# 1. رفع نسخة LoRA الكاملة
#/*ملاحظة غير الحساب وحط حسابك*/ username --> your huggingface username
print("🚀 جاري رفع نسخة LoRA الكاملة...")
model.push_to_hub("username/scientific-assistant-lora", token = True)
tokenizer.push_to_hub("username/scientific-assistant-lora", token = True)

# 2. رفع نسخة GGUF المختصرة (مباشرة من Unsloth)
print("🚀 جاري رفع نسخة GGUF المختصرة...")
model.push_to_hub_gguf(
    repo_id = "username/scientific-assistant-gguf", 
    tokenizer = tokenizer,
    quantization_method = "q4_k_m",
    token = True,
)

print("🎉 مبروك يا مستخدم! كل النسخ الآن مرفوعة على حسابك username")

SyntaxError: invalid syntax (1935150718.py, line 9)